In [1]:
import pandas as pd
import requests

canarias = pd.read_csv("../data/raw/canarias_pm10_samplingpoints.csv")
print(len(canarias), "puntos de medición")

API = "https://eeadmz1-downloads-api-appservice.azurewebsites.net"

body = {
    "countries": ["ES"],
    "cities": [],
    "pollutants": ["PM10"],
    "dataset": 1,
    "source": "API",
}

r = requests.post(f"{API}/ParquetFile/urls", json=body, timeout=120)
print(r.status_code)

60 puntos de medición
200


In [2]:
from io import StringIO

# 1. La respuesta de la API → tabla de URLs
text = r.content.decode("utf-8-sig")
urls = pd.read_csv(StringIO(text))

# 2. Nombre de archivo y código de estación, sacados de cada URL
urls["file_name"] = urls["ParquetFileUrl"].str.split("/").str[-1]
urls["station_code"] = urls["file_name"].str.split("_").str[1]

# 3. Canarias = códigos que empiezan con 35 o 38
is_canarias = urls["station_code"].str.startswith(("35", "38"))
urls_canarias = urls[is_canarias]

print(len(urls), "URLs de toda España")
print(len(urls_canarias), "URLs de Canarias")
print(urls_canarias["station_code"].nunique(), "estaciones")
urls_canarias.head()

417 URLs de toda España
49 URLs de Canarias
49 estaciones


,ParquetFileUrl,file_name,station_code
137,https://eeadmz1batchservice02.blob.core.window...,SP_35002001_10_A.parquet,35002001
138,https://eeadmz1batchservice02.blob.core.window...,SP_35002003_10_A.parquet,35002003
139,https://eeadmz1batchservice02.blob.core.window...,SP_35004001_10_A.parquet,35004001
140,https://eeadmz1batchservice02.blob.core.window...,SP_35016010_10_A.parquet,35016010
141,https://eeadmz1batchservice02.blob.core.window...,SP_35016012_10_A.parquet,35016012


In [3]:
print(r.request.body)                         # lo que REALMENTE se mandó a la API

b'{"countries": ["ES"], "cities": [], "pollutants": ["PM10"], "dataset": 1, "source": "API"}'


In [4]:
print(urls_canarias["ParquetFileUrl"].iloc[0])   # una URL completa, sin cortar

https://eeadmz1batchservice02.blob.core.windows.net/airquality-p/ES/SP_35002001_10_A.parquet


In [5]:
print(r.request.body)                            # tiene que decir "dataset": 1
print(urls_canarias["ParquetFileUrl"].iloc[0])   # la carpeta debería cambiar (¿airquality-p-e2a?)

b'{"countries": ["ES"], "cities": [], "pollutants": ["PM10"], "dataset": 1, "source": "API"}'
https://eeadmz1batchservice02.blob.core.windows.net/airquality-p/ES/SP_35002001_10_A.parquet


In [6]:
from pathlib import Path

def download(url: str, folder: Path) -> Path:
    folder.mkdir(parents=True, exist_ok=True)
    file = folder / url.split("/")[-1]
    if not file.exists():
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        file.write_bytes(r.content)
    return file

In [7]:
is_station = urls_canarias["station_code"] == "35002001"   # 1. la máscara
station_urls = urls_canarias[is_station]                   # 2. aplicarla
url = station_urls["ParquetFileUrl"].iloc[0]               # 3. sacar el texto de la URL
print(url)

https://eeadmz1batchservice02.blob.core.windows.net/airquality-p/ES/SP_35002001_10_A.parquet


In [8]:
file = download(url, Path("../data/raw/eea_pm10_e2a"))
recent = pd.read_parquet(file)
print(recent["Start"].min(), "→", recent["Start"].max())

2026-01-01 01:00:00 → 2026-11-25 17:00:00


In [9]:
recent["Verification"].value_counts()

Verification
3    7889
Name: count, dtype: int64

In [10]:
recent["Validity"].value_counts()

Validity
 1    6680
-1    1209
Name: count, dtype: int64